In [2]:
!pip install -q accelerate

In [5]:
%%writefile /kaggle/working/train_ddp_accelerate.py
# =========================================================================
# TASK ARITHMETIC MERGING - progetto clearsky (Multi-GPU con accelerate)
# (star removal, image restoration a pixel morti/caldi, super-resolution)
#
# GRID SEARCH A DUE STADI:
#   Stadio 1 ("safe zone"): grid search completa sui dataset con
#       degradazione singola -> IR, SR, SU
#       (attenzione alla convenzione: nei DATASET "SR" = star removal e
#        "SU" = super resolution; nei PESI/task_vectors invece la chiave
#        "super_resolution" corrisponde al task super-resolution e la
#        chiave "star_removal" corrisponde alla rimozione delle stelle.
#        Il nome cartella "SR" NON coincide col nome peso "super_resolution":
#        cartella dataset "SR" -> task PESO "star_removal"
#        cartella dataset "SU" -> task PESO "super_resolution"
#        cartella dataset "IR" -> task PESO "image_restoration")
#   Stadio 2 (bilanciamento fine): si riparte SOLO dalle combinazioni di
#       lambda trovate "sicure" nello stadio 1 e le si rivaluta sui
#       dataset con degradazioni combinate -> SR_IR, SR_SU, IR_SU
#       per scegliere il bilanciamento finale.
#
# I dataset sono enormi: per questo motivo ogni stadio lavora solo su una
# frazione campionata deterministicamente (subsample_pct), sia per lo
# stadio 1 che per lo stadio 2 (due percentuali separate, configurabili).
# =========================================================================

import os
import math
import glob
import json
import random
import itertools
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from accelerate import Accelerator
from accelerate.utils import set_seed


# =========================================================================
# ARCHITETTURA (invariata)
# =========================================================================

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()

        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        c = base_channels

        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)

        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)

        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)

        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)

        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)

        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)

        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)

        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)

        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())

        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)

        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)

        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)

        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)

        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)

        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)

        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)

        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)

        return self.out(h)


def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=300, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps

        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        sqrt_alpha_bar = torch.sqrt(alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', sqrt_alpha_bar)
        self.register_buffer('sqrt_one_minus_alpha_bar', sqrt_one_minus_alpha_bar)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise

    def forward(self, x_0, condition):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)

        x_t = self.forward_diffusion(x_0, t, noise)

        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)

        v_target = sqrt_alpha_bar_t * noise - sqrt_one_minus_alpha_bar_t * x_0
        predicted_v = self.network(x_t, t.float(), condition)

        loss = F.mse_loss(predicted_v, v_target)
        return loss

    @torch.no_grad()
    def sample(self, condition):
        """Sampling DDPM completo (ancestral)."""
        self.network.eval()
        device = self.beta.device

        batch_size, channels, height, width = condition.shape
        x = torch.randn(batch_size, channels, height, width, device=device)

        for i in reversed(range(self.n_steps)):
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = (sqrt_alpha_bar_t * x) - (sqrt_one_minus_alpha_bar_t * predicted_v)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            beta_t = self.beta[i]
            alpha_bar_prev_t = self.alpha_bar_prev[i]
            alpha_bar_t = self.alpha_bar[i]
            alpha_t = self.alpha[i]

            mean = (beta_t * torch.sqrt(alpha_bar_prev_t) / (1.0 - alpha_bar_t)) * pred_x0 + \
                   ((1.0 - alpha_bar_prev_t) * torch.sqrt(alpha_t) / (1.0 - alpha_bar_t)) * x

            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(self.posterior_variance[i])
                x = mean + sigma_t * noise
            else:
                x = mean

        self.network.train()
        return x

    @torch.no_grad()
    def sample_ddim(self, condition, ddim_steps=25, eta=0.0):
        """Sampling DDIM veloce, utile per test manuali rapidi prima del check finale a piena qualità."""
        self.network.eval()
        device = self.beta.device
        batch_size, channels, h, w = condition.shape
        x = torch.randn(batch_size, channels, h, w, device=device)

        step_indices = torch.linspace(0, self.n_steps - 1, ddim_steps, dtype=torch.long)
        step_indices = torch.unique(step_indices).flip(0)

        for idx in range(len(step_indices)):
            i = step_indices[idx].item()
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            alpha_bar_t = self.alpha_bar[i]
            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = sqrt_alpha_bar_t * x - sqrt_one_minus_alpha_bar_t * predicted_v
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            if idx == len(step_indices) - 1:
                x = pred_x0
                break

            i_prev = step_indices[idx + 1].item()
            alpha_bar_prev = self.alpha_bar[i_prev]

            pred_noise = (x - sqrt_alpha_bar_t * pred_x0) / (sqrt_one_minus_alpha_bar_t + 1e-8)

            if eta > 0.0:
                sigma_t = eta * torch.sqrt(
                    (1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev)
                )
                noise = torch.randn_like(x)
            else:
                sigma_t = torch.tensor(0.0, device=device)
                noise = torch.zeros_like(x)

            dir_xt = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma_t ** 2, min=0.0)) * pred_noise
            x = torch.sqrt(alpha_bar_prev) * pred_x0 + dir_xt + sigma_t * noise

        self.network.train()
        return x


# -------------------------------------------------------------------------
# CONFIG (modificato: grid search a due stadi + dataset separati)
# -------------------------------------------------------------------------
CONFIG = {
    "theta0_path": "/kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth",

    "checkpoints": {
        "star_removal":      "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100.pth",
        "image_restoration": "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_IR.pth",
        "super_resolution":  "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_SR.pth",
    },

    "ckpt_key": "ema_model",

    "in_channels": 3,
    "cond_channels": 3,
    "base_channels": 64,
    "n_steps": 200,

    # -----------------------------------------------------------------
    # Cartella base condivisa da tutti i dataset di valutazione
    # -----------------------------------------------------------------
    "dataset_base": "/kaggle/input/datasets/phantasm04/merged/dataset_merged",

    # STADIO 1: dataset con degradazione singola -> serve a individuare
    # la "safe zone" di combinazioni lambda che funzionano bene su ciascun
    # task preso isolatamente.
    # ATTENZIONE alla convenzione dei nomi cartella (diversa da quella dei
    # pesi): cartella "SR" = star removal, cartella "SU" = super resolution.
    "stage1_folders": ["IR", "SR", "SU"],

    # STADIO 2: dataset con degradazioni combinate a coppie -> usato per
    # trovare il bilanciamento fine SOLO fra le combinazioni lambda
    # sopravvissute allo stadio 1.
    "stage2_folders": ["SR_IR", "SR_SU", "IR_SU","SR_IR_SU"],

    # Percentuale dei dati totali da usare in ciascuno stadio (i dataset
    # sono enormi, quindi si campiona solo una frazione, in modo
    # deterministico e allineato fra i processi DDP)
    "stage1_subsample_pct": 0.05,
    "stage2_subsample_pct": 0.05,

    # Quante combinazioni (le migliori per val_loss) portare dallo
    # stadio 1 allo stadio 2 come "safe zone"
    "stage1_safe_zone_size": 10,

    # Grid search sui coefficienti di task arithmetic (usata SOLO nello
    # stadio 1; lo stadio 2 riusa esclusivamente le combinazioni della
    # safe zone trovata nello stadio 1)
    "lambda_grid_values": {
        "star_removal":      [0.8, 0.9, 1.0],
        "image_restoration":  [0.1, 0.3, 0.5],
        "super_resolution":   [0.0, 0.1, 0.3],
    },

    "use_ddim_for_manual_tests": True,
    "manual_test_ddim_steps": 25,
    "manual_test_ddim_eta": 0.0,

    "eval_batch_size": 8,

    # Impostato a None per valutare l'intera fetta campionata tramite subsample_pct
    "eval_max_samples_per_task": None,

    "output_path": "/kaggle/working/ddpm_merged_task_arithmetic.pth",

    # Quante migliori combinazioni stampare nel riepilogo finale di ogni stadio
    "grid_search_top_k": 15,
}


# =========================================================================
# UTILITY (invariata)
# =========================================================================

def load_unet_state_dict(path: str, key: str = None):
    obj = torch.load(path, map_location="cpu")
    sd = obj[key] if key is not None else obj

    if any(k.startswith("_orig_mod.") for k in sd.keys()):
        sd = {k[len("_orig_mod."):]: v for k, v in sd.items()}

    if any(k.startswith("network.") for k in sd.keys()):
        sd = {k[len("network."):]: v for k, v in sd.items() if k.startswith("network.")}

    return OrderedDict(sd)


def assert_compatible(reference, checkpoints: dict):
    ref_keys = set(reference.keys())
    for name, sd in checkpoints.items():
        keys = set(sd.keys())
        if keys != ref_keys:
            raise ValueError(
                f"[{name}] chiavi non compatibili con theta0.\n"
                f"  mancanti: {ref_keys - keys}\n  in eccesso: {keys - ref_keys}"
            )
        for k in ref_keys:
            if sd[k].shape != reference[k].shape:
                raise ValueError(f"[{name}] shape mismatch su '{k}': {sd[k].shape} vs {reference[k].shape}")


def compute_task_vector(theta_t, theta_0):
    tau = OrderedDict()
    for k, v0 in theta_0.items():
        if torch.is_floating_point(v0):
            tau[k] = theta_t[k].float() - v0.float()
        else:
            tau[k] = torch.zeros_like(v0)
    return tau


def task_vector_norm(tau) -> float:
    return torch.sqrt(sum((v ** 2).sum() for v in tau.values() if torch.is_floating_point(v))).item()


def merge_task_arithmetic(theta_0, task_vectors: dict, lambdas: dict):
    merged = OrderedDict()
    for k, v0 in theta_0.items():
        if torch.is_floating_point(v0):
            acc = v0.float().clone()
            for name, tau in task_vectors.items():
                acc = acc + lambdas[name] * tau[k]
            merged[k] = acc.to(v0.dtype)
        else:
            merged[k] = v0.clone()
    return merged


# =========================================================================
# DATASET DI VALUTAZIONE
# =========================================================================

class DegradedImageDataset(Dataset):
    def __init__(self, roots: list, subsample_pct: float = 0.1, normalize_to_minus1_1: bool = True, seed: int = 42):
        self.normalize = normalize_to_minus1_1
        self.samples = []

        # Raccoglie in modo aggregato tutte le coppie input-target da ciascun percorso sorgente
        for root in roots:
            input_dir = os.path.join(root, "input", "npy")
            target_dir = os.path.join(root, "target", "npy")

            if not os.path.exists(input_dir) or not os.path.exists(target_dir):
                continue

            all_files = sorted(
                os.path.basename(p) for p in glob.glob(os.path.join(input_dir, "*.npy"))
            )

            for f in all_files:
                in_path = os.path.join(input_dir, f)
                tgt_path = os.path.join(target_dir, f)
                if os.path.exists(tgt_path):
                    self.samples.append((in_path, tgt_path))

        if not self.samples:
            raise FileNotFoundError(f"Nessun file .npy accoppiato trovato nei percorsi forniti: {roots}")

        # Campionamento deterministico basato sulla percentuale (cruciale per allineare i processi in DDP)
        self.samples.sort()  # Forza ordinamento alfabetico stabile
        rng = random.Random(seed)
        num_samples = max(1, int(len(self.samples) * subsample_pct))
        self.samples = rng.sample(self.samples, num_samples)
        self.samples.sort()  # Riordina per efficienza di lettura sequenziale

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        in_path, tgt_path = self.samples[idx]

        x = np.load(in_path).astype(np.float32)
        y = np.load(tgt_path).astype(np.float32)

        x = torch.from_numpy(x)
        y = torch.from_numpy(y)

        if x.ndim == 3 and x.shape[-1] in (1, 3):
            x = x.permute(2, 0, 1)
        if y.ndim == 3 and y.shape[-1] in (1, 3):
            y = y.permute(2, 0, 1)

        if self.normalize:
            x = (x * 2.0) - 1.0
            y = (y * 2.0) - 1.0

        return x, y


def build_per_folder_datasets(dataset_base: str, folder_names: list, subsample_pct: float,
                               accelerator: Accelerator, seed: int = 42) -> dict:
    """
    Costruisce UN dataset per ciascuna cartella (senza fonderle insieme), così
    da poter stampare una breakdown separata per cartella durante la valutazione
    (ad es. IR / SR / SU nello stadio 1, oppure SR_IR / SR_SU / IR_SU nello stadio 2).
    Ogni dataset viene campionato in modo deterministico alla percentuale richiesta,
    perché le cartelle sorgente sono troppo grandi per essere usate per intero.
    """
    datasets = {}
    for folder in folder_names:
        root = os.path.join(dataset_base, folder)
        accelerator.print(f"  Caricamento cartella '{folder}' da {root} ...")
        ds = DegradedImageDataset(roots=[root], subsample_pct=subsample_pct, seed=seed)
        accelerator.print(f"    -> {len(ds)} campioni estratti ({subsample_pct * 100:.1f}% dei totali).")
        datasets[folder] = ds
    return datasets


def prepare_loaders(datasets_dict: dict, batch_size: int, accelerator: Accelerator) -> dict:
    prepared = {}
    for name, ds in datasets_dict.items():
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        prepared[name] = accelerator.prepare(loader)
    return prepared


# =========================================================================
# BUILD MODEL / EVALUATE
# =========================================================================

def build_model_fn(state_dict, config=CONFIG):
    unet = ImprovedUNet(
        in_channels=config["in_channels"],
        cond_channels=config["cond_channels"],
        base_channels=config["base_channels"],
    )
    unet.load_state_dict(state_dict)
    unet.eval()

    ddpm = DDPMvPrediction(unet, n_steps=config["n_steps"])
    ddpm.eval()

    return ddpm


@torch.no_grad()
def evaluate_fn(model, prepared_loaders: dict, accelerator: Accelerator,
                 max_samples_per_task: int = None,
                 use_ddim: bool = False, ddim_steps: int = 25, ddim_eta: float = 0.0):
    set_seed(42)

    max_samples_per_task = max_samples_per_task if max_samples_per_task is not None else CONFIG["eval_max_samples_per_task"]

    per_task_mse = {}
    all_mse = []

    for task_name, loader in prepared_loaders.items():
        task_mse = []
        n_seen = 0

        for x, y in loader:
            unwrapped_model = accelerator.unwrap_model(model)

            if use_ddim:
                pred_y = unwrapped_model.sample_ddim(condition=x, ddim_steps=ddim_steps, eta=ddim_eta)
            else:
                pred_y = unwrapped_model.sample(condition=x)

            loss_per_sample = F.mse_loss(pred_y, y, reduction='none').mean(dim=[1, 2, 3])
            gathered_losses = accelerator.gather_for_metrics(loss_per_sample)

            task_mse.extend(gathered_losses.cpu().tolist())
            n_seen += len(gathered_losses)

            if max_samples_per_task is not None and n_seen >= max_samples_per_task:
                break

        avg = sum(task_mse) / len(task_mse) if len(task_mse) > 0 else 0.0
        per_task_mse[task_name] = avg
        all_mse.extend(task_mse)

        accelerator.print(f"    [{task_name:30s}] MSE = {avg:.6f}  (n={len(task_mse)})")

    overall = sum(all_mse) / len(all_mse) if len(all_mse) > 0 else 0.0
    evaluate_fn.last_breakdown = per_task_mse
    return overall


# =========================================================================
# GRID SEARCH (funzione unica, riusata sia per lo stadio 1 che per lo stadio 2)
# =========================================================================

def generate_lambda_grid(task_names, grid_values):
    lists_of_values = [grid_values[name] for name in task_names]
    for combo in itertools.product(*lists_of_values):
        # USA YIELD, NON RETURN!
        yield dict(zip(task_names, combo))


def run_lambda_search(theta_0, task_vectors, lambda_combos: list,
                       build_model_fn, evaluate_fn_wrapper, accelerator,
                       top_k: int = 15, stage_label: str = ""):
    """
    Valuta esplicitamente la lista di combinazioni lambda fornita (non la
    ricostruisce da una grid_values): questo permette di riusare la stessa
    funzione sia per la grid search completa dello stadio 1 sia per la
    ri-valutazione ristretta alla "safe zone" nello stadio 2.
    """
    total = len(lambda_combos)
    accelerator.print(f"\n[{stage_label}] {total} combinazioni di lambda da testare.\n")

    results = []
    best = None

    for i, lambdas in enumerate(lambda_combos, start=1):
        merged_sd = merge_task_arithmetic(theta_0, task_vectors, lambdas)
        model = build_model_fn(merged_sd)
        model = model.to(accelerator.device)

        lam_str = ", ".join(f"{k}={v:.2f}" for k, v in lambdas.items())
        accelerator.print(f"  --- [{stage_label} {i}/{total}] {lam_str} ---")

        score = evaluate_fn_wrapper(model)
        breakdown = dict(evaluate_fn.last_breakdown)
        accelerator.print(f"      val_loss={score:.6f}")

        results.append((lambdas, score, breakdown))
        if best is None or score < best[1]:
            best = (lambdas, score, breakdown)
            accelerator.print(f"      >>> Nuovo migliore! val_loss={score:.6f}")

        del model
        accelerator.free_memory()
        torch.cuda.empty_cache()

    results_sorted = sorted(results, key=lambda r: r[1])

    accelerator.print("\n" + "=" * 70)
    accelerator.print(f"[{stage_label}] RIEPILOGO — top {min(top_k, total)} su {total} combinazioni testate:")
    for lambdas, score, breakdown in results_sorted[:top_k]:
        lam_str = ", ".join(f"{k}={v:.2f}" for k, v in lambdas.items())
        accelerator.print(f"  val_loss={score:.6f}  |  {lam_str}")
        accelerator.print(f"    breakdown per cartella: {breakdown}")
    accelerator.print("=" * 70)

    best_lambdas, best_score, best_breakdown = results_sorted[0]
    accelerator.print(f"\n[{stage_label}] Migliore trovata: {best_lambdas} (val_loss={best_score:.6f})")
    return best_lambdas, results_sorted


# =========================================================================
# MAIN
# =========================================================================

def main(config=CONFIG):
    accelerator = Accelerator()

    # --- 1. theta0 ---
    theta0 = load_unet_state_dict(config["theta0_path"])
    accelerator.print(f"[theta0] caricato da {config['theta0_path']}")

    # --- 2. checkpoint dei 3 task ---
    checkpoints = {
        name: load_unet_state_dict(path, key=config["ckpt_key"])
        for name, path in config["checkpoints"].items()
    }
    assert_compatible(theta0, checkpoints)
    accelerator.print("Tutti i checkpoint sono compatibili con theta0 (stesse chiavi/shape).")

    # --- 3. task vector + diagnostica sulle norme ---
    task_vectors = {name: compute_task_vector(sd, theta0) for name, sd in checkpoints.items()}
    accelerator.print("\nNorme dei task vector (||theta_t - theta_0||):")
    for name, tau in task_vectors.items():
        accelerator.print(f"  {name:20s}: {task_vector_norm(tau):.4f}")

    def manual_eval_wrapper(model, prepared_loaders):
        return evaluate_fn(
            model, prepared_loaders, accelerator,
            use_ddim=config["use_ddim_for_manual_tests"],
            ddim_steps=config["manual_test_ddim_steps"],
            ddim_eta=config["manual_test_ddim_eta"],
        )

    # =====================================================================
    # STADIO 1: grid search completa su IR / SR / SU -> individua la "safe zone"
    # =====================================================================
    accelerator.print("\n" + "#" * 70)
    accelerator.print("# STADIO 1: grid search su dataset a degradazione singola (IR, SR, SU)")
    accelerator.print("#" * 70)

    stage1_datasets = build_per_folder_datasets(
        config["dataset_base"], config["stage1_folders"],
        config["stage1_subsample_pct"], accelerator,
    )
    stage1_loaders = prepare_loaders(stage1_datasets, config["eval_batch_size"], accelerator)

    task_names = list(task_vectors.keys())
    stage1_combos = list(generate_lambda_grid(task_names, config["lambda_grid_values"]))

    stage1_best_lambdas, stage1_results = run_lambda_search(
        theta0, task_vectors, stage1_combos,
        build_model_fn=lambda sd: build_model_fn(sd, config=config),
        evaluate_fn_wrapper=lambda model: manual_eval_wrapper(model, stage1_loaders),
        accelerator=accelerator,
        top_k=config["grid_search_top_k"],
        stage_label="STADIO 1",
    )

    safe_zone_size = min(config["stage1_safe_zone_size"], len(stage1_results))
    safe_zone_lambdas = [lambdas for lambdas, _, _ in stage1_results[:safe_zone_size]]

    accelerator.print(f"\n'Safe zone' selezionata: le {safe_zone_size} migliori combinazioni dello stadio 1:")
    for lambdas in safe_zone_lambdas:
        lam_str = ", ".join(f"{k}={v:.2f}" for k, v in lambdas.items())
        accelerator.print(f"  {lam_str}")

    # Libera memoria/loader dello stadio 1 prima di passare allo stadio 2
    del stage1_loaders, stage1_datasets
    accelerator.free_memory()
    torch.cuda.empty_cache()

    # =====================================================================
    # STADIO 2: ri-valutazione della safe zone su SR_IR / SR_SU / IR_SU
    # per trovare il bilanciamento finale
    # =====================================================================
    accelerator.print("\n" + "#" * 70)
    accelerator.print("# STADIO 2: bilanciamento fine su dataset a degradazione combinata (SR_IR, SR_SU, IR_SU)")
    accelerator.print("#" * 70)

    stage2_datasets = build_per_folder_datasets(
        config["dataset_base"], config["stage2_folders"],
        config["stage2_subsample_pct"], accelerator,
    )
    stage2_loaders = prepare_loaders(stage2_datasets, config["eval_batch_size"], accelerator)

    best_lambdas, stage2_results = run_lambda_search(
        theta0, task_vectors, safe_zone_lambdas,
        build_model_fn=lambda sd: build_model_fn(sd, config=config),
        evaluate_fn_wrapper=lambda model: manual_eval_wrapper(model, stage2_loaders),
        accelerator=accelerator,
        top_k=config["grid_search_top_k"],
        stage_label="STADIO 2",
    )

    del stage2_loaders, stage2_datasets
    accelerator.free_memory()
    torch.cuda.empty_cache()

    # --- 6. merge finale col miglior set trovato nello stadio 2 ---
    accelerator.print(f"\nLambda finali usati (migliori dello stadio 2, dentro la safe zone dello stadio 1): {best_lambdas}")
    merged_state_dict = merge_task_arithmetic(theta0, task_vectors, best_lambdas)

    n_bad = sum(int(not torch.isfinite(v).all()) for v in merged_state_dict.values() if torch.is_floating_point(v))
    if n_bad:
        raise RuntimeError(f"{n_bad} tensori del modello fuso contengono NaN/Inf, controlla i checkpoint di input.")

    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        torch.save(merged_state_dict, config["output_path"])
        accelerator.print(f"\nModello fuso (Task Arithmetic) salvato in: {config['output_path']}")
        accelerator.print("Per usarlo: "
             f"unet = ImprovedUNet(in_channels={config['in_channels']}, "
             f"cond_channels={config['cond_channels']}, base_channels={config['base_channels']}); "
             f"unet.load_state_dict(torch.load('{config['output_path']}'))")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/train_ddp_accelerate.py


In [6]:
!accelerate launch --num_processes=2 /kaggle/working/train_ddp_accelerate.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[theta0] caricato da /kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth
Tutti i checkpoint sono compatibili con theta0 (stesse chiavi/shape).

Norme dei task vector (||theta_t - theta_0||):
  star_removal        : 56.4456
  image_restoration   : 50.3134
  super_resolution    : 80.9864

######################################################################
# STADIO 1: grid search su dataset a degradazione singola (IR, SR, SU)
######################################################################
  Caricamento cartella 'IR

In [10]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os

# Importiamo le classi necessarie dallo script appena creato
from train_ddp_accelerate import ImprovedUNet, DDPMvPrediction, DegradedImageDataset

# =========================================================================
# CONFIGURAZIONE
# =========================================================================
MODEL_PATH = "/kaggle/working/ddpm_merged_task_arithmetic.pth"

# ATTENZIONE: Scegli la sottocartella specifica del task combinato che vuoi testare
# (es. la cartella contenente "star_removal_image_restoration" o simili)
DATASET_DIR = "/kaggle/input/datasets/phantasm04/star-removal-uar/dataset" 

INDICES_TO_VISUALIZE = [30, 70, 115, 130, 150, ]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================================
# CARICAMENTO MODELLO E DATASET
# =========================================================================
print("Caricamento del modello fuso in corso...")
unet = ImprovedUNet(in_channels=3, cond_channels=3, base_channels=64)
unet.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
unet.to(DEVICE)
unet.eval()

ddpm = DDPMvPrediction(unet, n_steps=200).to(DEVICE)
ddpm.eval()

print(f"Caricamento dataset da: {DATASET_DIR}")
dataset = DegradedImageDataset(DATASET_DIR, normalize_to_minus1_1=True)

def denormalize(t):
    """Riporta i tensori dal range [-1, 1] a [0, 1] per matplotlib."""
    t = (t + 1.0) / 2.0
    return torch.clamp(t, 0.0, 1.0)

# =========================================================================
# INFERENZA E PLOT
# =========================================================================
n_images = len(INDICES_TO_VISUALIZE)
fig, axes = plt.subplots(n_images, 3, figsize=(15, 5 * n_images))

# Gestione per il caso in cui ci sia un solo indice
if n_images == 1:
    axes = [axes]

print("Inizio inferenza e generazione immagini...")
with torch.no_grad():
    for i, idx in enumerate(INDICES_TO_VISUALIZE):
        if idx >= len(dataset):
            print(f"[Warning] L'indice {idx} supera la dimensione del dataset ({len(dataset)}). Salto.")
            continue
            
        cond, target = dataset[idx]
        
        # Aggiungiamo la dimensione del batch (B, C, H, W) e spostiamo su GPU
        cond_batch = cond.unsqueeze(0).to(DEVICE)
        
        # Generiamo la predizione usando DDIM (molto più veloce per visualizzare)
        print(f"Campionamento per l'indice {idx} (DDIM)...")
        pred = ddpm.sample_ddim(condition=cond_batch, ddim_steps=25, eta=0.0)
        
        # Se preferisci la massima fedeltà e non ti importa del tempo di attesa, 
        # commenta la riga sopra e de-commenta la seguente:
        # pred = ddpm.sample(condition=cond_batch)
        
        # Prepariamo i tensori per il plotting: (C, H, W) -> (H, W, C)
        cond_img = denormalize(cond).cpu().permute(1, 2, 0).numpy()
        target_img = denormalize(target).cpu().permute(1, 2, 0).numpy()
        pred_img = denormalize(pred.squeeze(0)).cpu().permute(1, 2, 0).numpy()
        
        # Plot Input (Condition)
        axes[i, 0].imshow(cond_img)
        axes[i, 0].set_title(f"Input Degradato (Idx: {idx})", fontsize=12)
        axes[i, 0].axis('off')
        
        # Plot Target
        axes[i, 1].imshow(target_img)
        axes[i, 1].set_title(f"Target Ground Truth (Idx: {idx})", fontsize=12)
        axes[i, 1].axis('off')
        
        # Plot Prediction
        axes[i, 2].imshow(pred_img)
        axes[i, 2].set_title(f"Prediction (Modello Fuso)", fontsize=12)
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

Caricamento del modello fuso in corso...
Caricamento dataset da: /kaggle/input/datasets/phantasm04/star-removal-uar/dataset


FileNotFoundError: Nessun file .npy accoppiato trovato nei percorsi forniti: /kaggle/input/datasets/phantasm04/star-removal-uar/dataset

In [17]:
import os
print(os.listdir("/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR_IR_SU/input/"))

['npy']
